In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

ROOT = Path.cwd()
DATA_PATH = ROOT / "outputs" / "cleaned" / "songs_pre_imputation.parquet"
TARGET = "energy"

FEATURE_POOL = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "tempo",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "key",
    "mode",
]

feature_columns = [column for column in FEATURE_POOL if column != TARGET]
best_params = {
    "n_estimators": 300,
    "max_depth": 10,
    "learning_rate": 0.08960785365368121,
    "subsample": 0.8394633936788146,
    "colsample_bytree": 0.6624074561769746,
    "min_child_weight": 2.403950683025824,
    "reg_alpha": 3.3323645788192616e-08,
    "reg_lambda": 0.6245760287469887,
    "gamma": 0.002570603566117596,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}


In [ ]:
columns_to_load = sorted(set(FEATURE_POOL + ["track_id", "artist_name", "track_name"]))
df = pd.read_parquet(DATA_PATH, columns=columns_to_load)
df.shape


In [ ]:
observed_df = df[df[TARGET].notna()].copy()
missing_df = df[df[TARGET].isna()].copy()

X_observed = observed_df[feature_columns]
y_observed = observed_df[TARGET]
X_missing = missing_df[feature_columns]

X_train, X_test, y_train, y_test = train_test_split(
    X_observed,
    y_observed,
    test_size=0.2,
    random_state=42,
)

print(len(observed_df), len(missing_df))


In [ ]:
model = XGBRegressor(**best_params)
model.fit(X_train, y_train)

test_predictions = model.predict(X_test)
test_predictions = np.clip(test_predictions, 0, 1)

test_metrics = {
    "mae": mean_absolute_error(y_test, test_predictions),
    "rmse": np.sqrt(mean_squared_error(y_test, test_predictions)),
    "r2": r2_score(y_test, test_predictions),
}

test_metrics


In [ ]:
final_model = XGBRegressor(**best_params)
final_model.fit(X_observed, y_observed)

predictions = final_model.predict(X_missing)
predictions = np.clip(predictions, 0, 1)


In [ ]:
pd.Series(test_predictions).describe()


In [ ]:
pd.Series(test_predictions).nunique()


In [ ]:
pd.Series(predictions).describe()


In [ ]:
pd.Series(predictions).nunique()


In [ ]:
X_missing.nunique().sort_values()
